In [41]:
import pandas as pd

In [42]:
ICD_CODES = ["C61", "C34", "C50", "C25", "C22", "C18", "C64", "C80", "C83", "C91"]

In [43]:
WAY_TO_MERGE = "forward"
MODEL = "GPT_OSS_MODEL_SIZE.SMALL_Low_w_rnd_lvl"
TRY = 0
scores = pd.read_csv(f"scores/scores_{MODEL}_{TRY}_{WAY_TO_MERGE}.csv").rename(columns={"Unnamed: 0": "icd10_category"})
scores = scores[["icd10_category", *ICD_CODES]]

In [44]:
scores.head()

,icd10_category,C61,C34,C50,C25,C22,C18,C64,C80,C83,C91
0,A00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,A01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,A02,0.0,-1.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,A04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:
scores = scores.melt(
    id_vars=["icd10_category"], 
    var_name="icd10_category_1", 
    value_name="score"
).rename(columns={"icd10_category": "icd10_category_2"})

In [46]:
scores.head()

,icd10_category_2,icd10_category_1,score
0,A00,C61,0.0
1,A01,C61,0.0
2,A02,C61,0.0
3,A03,C61,0.0
4,A04,C61,0.0


In [47]:
scores["score"].value_counts()

score
 0.0    18099
-1.0     1157
 1.0      224
Name: count, dtype: int64

In [48]:
MODEL = "GPT_OSS_MODEL_SIZE.SMALL_Low_w_rnd_lvl"
TRY = 0
# WAY_TO_MERGE = "mean"
WAY_TO_MERGE = "forward"
scores_mcq = pd.read_csv(f"scores_mcq/scores_{MODEL}_{','.join(ICD_CODES)}_{TRY}_{WAY_TO_MERGE}.csv").drop("Unnamed: 0", axis=1, errors="ignore")

In [49]:
scores_mcq.head()

,icd10_category_1,icd10_category_2,score
0,C61,K76,-1
1,C61,R18,1
2,C61,K74,0
3,C61,B19,-1
4,C61,J44,-1


In [50]:
scores_mcq['score'].value_counts()

score
 0    9646
 1    3978
-1    3336
Name: count, dtype: int64

In [51]:
all_scores = pd.merge(
    scores,
    scores_mcq,
    how="inner",
    left_on=["icd10_category_1", "icd10_category_2"],
    right_on=["icd10_category_1", "icd10_category_2"],
    suffixes=("", "_mcq")
)

In [52]:
all_scores.shape

(16960, 4)

In [53]:
all_scores.head(2)

,icd10_category_2,icd10_category_1,score,score_mcq
0,A01,C61,0.0,0
1,A02,C61,0.0,0


In [54]:
all_scores["is_equal"] = (all_scores["score"] == all_scores["score_mcq"]).astype(int)

In [55]:
all_scores["is_equal"].sum()/len(all_scores)

0.5433372641509434

In [56]:
all_scores_no_zero = all_scores[(all_scores["score"] != 0) | (all_scores["score_mcq"] != 0)]
all_scores_no_zero["is_equal"].sum()/len(all_scores_no_zero)

0.04110437043456729

In [57]:
import sklearn
import sklearn.metrics 

### average metrics over all dataset

In [58]:
for hard_metric in ["accuracy", "f1", "precision", "recall"]:
    print(
        hard_metric, ":", 
        round(getattr(sklearn.metrics, hard_metric+"_score")(
            all_scores['score_mcq'] >= 0.5, 
            all_scores['score'] >= 0.5
        ), 4)
    )

for soft_metric in ["roc_auc", "average_precision"]:
    print(
        soft_metric, ":", 
        round(getattr(sklearn.metrics, soft_metric+"_score")(
            all_scores['score_mcq'] >= 0.5, 
            all_scores['score']
        ), 4)
    )

accuracy : 0.767
f1 : 0.0595
precision : 0.558
recall : 0.0314
roc_auc : 0.5218
average_precision : 0.2486


### average metrics by ICD-10 category

In [59]:
results = []

for icd_code, icd_code_df in all_scores.groupby("icd10_category_1"):
    idc_code_result = {"icd_code": icd_code}
    for hard_metric in ["accuracy", "f1", "precision", "recall"]:
        idc_code_result[hard_metric] = round(
            getattr(sklearn.metrics, hard_metric+"_score")(
                icd_code_df['score_mcq'] >= 0.5, 
                icd_code_df['score'] >= 0.5
            ), 
            4
        )

    for soft_metric in ["roc_auc", "average_precision"]:
        idc_code_result[soft_metric] = round(
            getattr(sklearn.metrics, soft_metric+"_score")(
                icd_code_df['score_mcq'] >= 0.5, 
                icd_code_df['score']
            ), 
            4
        )

    icd_code_df_no_zero = icd_code_df[(icd_code_df["score"] != 0) | (icd_code_df["score_mcq"] != 0)]
    icd_code_df_no_zero["is_equal"] = (icd_code_df_no_zero["score"] == icd_code_df_no_zero["score_mcq"]).astype(int)
    idc_code_result["intersection_of_!=0_elements"] = round(icd_code_df_no_zero["is_equal"].sum()/len(icd_code_df_no_zero), 4)

    results.append(idc_code_result)

results_df = pd.DataFrame(results)
results_df

/tmp/ipykernel_488815/4229212794.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  icd_code_df_no_zero["is_equal"] = (icd_code_df_no_zero["score"] == icd_code_df_no_zero["score_mcq"]).astype(int)
/tmp/ipykernel_488815/4229212794.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  icd_code_df_no_zero["is_equal"] = (icd_code_df_no_zero["score"] == icd_code_df_no_zero["score_mcq"]).astype(int)
/tmp/ipykernel_488815/4229212794.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice 

,icd_code,accuracy,f1,precision,recall,roc_auc,average_precision,intersection_of_!=0_elements
0,C18,0.8131,0.0704,0.4615,0.0381,0.5211,0.1986,0.0416
1,C22,0.7783,0.0647,0.5652,0.0343,0.5255,0.2396,0.0333
2,C25,0.7842,0.0469,0.6429,0.0243,0.5225,0.2328,0.0293
3,C34,0.7400,0.1370,0.6364,0.0768,0.5468,0.3048,0.0751
4,C50,0.8154,0.1083,0.5588,0.0599,0.5432,0.2158,0.0789
5,C61,0.8496,0.0659,0.4737,0.0354,0.5147,0.1616,0.0436
6,C64,0.8467,0.0299,0.3333,0.0156,0.5146,0.1563,0.0183
7,C80,0.6680,0.0243,0.5833,0.0124,0.5138,0.3408,0.0201
8,C83,0.7276,0.0375,0.6000,0.0194,0.5159,0.2841,0.0339
9,C91,0.6468,0.0260,0.5714,0.0133,0.5035,0.3572,0.0296


In [60]:
mean_results = results_df.drop(columns=["icd_code"]).mean(axis=0).round(4)
mean_results

accuracy                        0.7670
f1                              0.0611
precision                       0.5426
recall                          0.0329
roc_auc                         0.5222
average_precision               0.2492
intersection_of_!=0_elements    0.0404
dtype: float64

In [61]:
pd.concat([results_df, pd.DataFrame.from_records([mean_results.to_dict()])], axis=0)

,icd_code,accuracy,f1,precision,recall,roc_auc,average_precision,intersection_of_!=0_elements
0,C18,0.8131,0.0704,0.4615,0.0381,0.5211,0.1986,0.0416
1,C22,0.7783,0.0647,0.5652,0.0343,0.5255,0.2396,0.0333
2,C25,0.7842,0.0469,0.6429,0.0243,0.5225,0.2328,0.0293
3,C34,0.7400,0.1370,0.6364,0.0768,0.5468,0.3048,0.0751
4,C50,0.8154,0.1083,0.5588,0.0599,0.5432,0.2158,0.0789
5,C61,0.8496,0.0659,0.4737,0.0354,0.5147,0.1616,0.0436
6,C64,0.8467,0.0299,0.3333,0.0156,0.5146,0.1563,0.0183
7,C80,0.6680,0.0243,0.5833,0.0124,0.5138,0.3408,0.0201
8,C83,0.7276,0.0375,0.6000,0.0194,0.5159,0.2841,0.0339
9,C91,0.6468,0.0260,0.5714,0.0133,0.5035,0.3572,0.0296
